### This is the code for Multi-Layer Perceptron

In [ ]:
import random
import numpy as np

In [ ]:
def sigmoid(z):
    """sigmoid function."""
    return 1.0/(1.0+np.exp(-z))

In [ ]:
def sigmoid_prime(z):
    """Derivative the sigmoid functions."""
    return sigmoid(z) * (1 - sigmoid(z))

In [ ]:
class Network(object):
    
    
    def __init__(self, sizes):
        """initialize the bias and weights"""
        self.num_layers = len(sizes)
        self.sizes = sizes
        self.biases = [np.random.randn(y, 1) for y in sizes [1:]]
        self.weights = [np.random.randn(y, x) for x, y in zip(sizes[:-1], sizes[1:])]

        

    def feedforward(self, a):
        """Return the output of the networks if 'a' is a input"""
        for b, w in zip(self.biases, self.weights):
            a = sigmoid(np.dot(w, a) + b)
        return a


    def SGD(self, training_data, epochs, mini_batch_size, eta, test_data=None):
        """Train the neural network using mini-batch stochastic gradient descent"""
        if test_data:
            n_test = len(test_data)
            n = len(training_data)
        for j in range(epochs):
            random.shuffle(training_data)
            mini_batches = [training_data[k:k+mini_batch_size] for k in range(0, n, mini_batch_size)]
            for mini_batch in mini_batches:
                self.update_mini_batch(mini_batch, eta)
            if test_data:
                print ("Epoch {0}: {1}/{2}".format(j, self.evaluate(test_data), n_test))
            else:
                print ("Epoch {0} complete".fornat(j))


    def update_mini_batch(self, mini_batch, eta):
        """update the network's weights and biases by applying gradient descent using backpropagation to a single minibatch."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
        for x, y in mini_batch:
            delta_nabla_b, delta_nabla_w = self.backprop(x, y)
            nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
            nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
        self.weights = [w - (eta/len(mini_batch))*nw for w, nw in zip(self.weights, nabla_w)]
        self.biases = [b - (eta/len(mini_batch))*nb for b, nb in zip(self.biases, nabla_b)]


    def backprop(self, x, y):
        """return a tuple''(nabla_b, nabla_w)''representing the gradient for the cost function C_x.''nabla_b''and ''nabla_w''are 
        layer-by-layer lists  of numpy arrays, similar to''self.biases''and ''self.weights''."""
        nabla_b = [np.zeros(b.shape) for b in self.biases]
        nabla_w = [np.zeros(w.shape) for w in self.weights]
    
        # feedforward
        activation = x
        activations = [x] # list to store all the activations, layer by layer
        zs = [] # list to store all the z vectors, layer by layer
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)
        
        # backward pass
        delta = self.cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1])
        nabla_b[-1] = delta
        nabla_w[-1] = np.dot(delta, activations[-2].transpose())
    
        # l = 1 means the last layer of neurons, l = 2 is the second-last layer, and so on.
        for l in range(2, self.num_layers):
            z = zs[-l]
            sp = sigmoid_prime(z)
            delta = np.dot(self.weights[-l+1].transpose(), delta) * sp
            nabla_b[-l] = delta
            nabla_w[-l] = np.dot(delta, activations[-l-1].transpose())
            return (nabla_b, nabla_w)


    def evaluate(self, test_data):
        """Return the number of the test inputs for which the neural networks outputs the correct result."""
        test_results = []
        for (x, y) in test_data:
            # 暴力拆包：不管 y 是数组还是数字，都强制转成纯整数
            if isinstance(y, np.ndarray):
                true_label = int(y.flatten()[0])
            else:
                true_label = int(y)
            
            # 获取网络预测的数字
            prediction = np.argmax(self.feedforward(x))
            
            # 把预测结果和真实标签存下来
            test_results.append((prediction, true_label))
            
            # 统计正确的数量
            return sum(int(x == y) for (x, y) in test_results)


    def cost_derivative(self, output_activations, y):
        """ Return the vector of partial derivative of a about C_x"""
        return (output_activations - y)

In [ ]:
"""cause i met some difficulties to loading the data ftom the website which 
author provided so i input training data and test data from MNIST directly. Code in this cell is from deepseek"""
import numpy as np
from sklearn.datasets import fetch_openml

print("Loading MNIST from scikit-learn (no download needed)...")

mnist = fetch_openml('mnist_784', version=1, cache=True, parser='auto')


X = mnist.data.values / 255.0
y = mnist.target.values.astype(int)


X_train, X_test = X[:50000], X[50000:]
y_train, y_test = y[:50000], y[50000:]


def format_data(X_data, y_data):
    result = []
    for img, label in zip(X_data, y_data):
       
        img_vector = img.reshape(-1, 1)
     
        label_vector = np.zeros((10, 1))
        label_vector[label] = 1.0
        result.append((img_vector, label_vector))
    return result

training_data = format_data(X_train, y_train)
test_data = format_data(X_test, y_test)


validation_data = training_data[:10000]
training_data = training_data[10000:]

print("Data loaded successfully!")
print("Training data size:", len(training_data))
print("Validation data size:", len(validation_data))
print("Test data size:", len(test_data))

"""cause the format of the training data doesnt match the code so we have to change it."""
import numpy as np

new_training_data = []
for img, label in training_data:
    img_vector = img.reshape(-1, 1)
    label_vector = np.zeros((10, 1))
    
    # --- 终极修复部分 ---
    # 提取纯数字（不管它是数组还是数字，都强制转成 int）
    if isinstance(label, np.ndarray):
        clean_label = int(label.flatten()[0])
    else:
        clean_label = int(label)
    # -------------------
    
    label_vector[clean_label] = 1.0
    new_training_data.append((img_vector, label_vector))

new_test_data = []
for img, label in test_data:
    img_vector = img.reshape(-1, 1)
    if isinstance(label, np.ndarray):
        clean_label = int(label.flatten()[0])
    else:
        clean_label = int(label)
    new_test_data.append((img_vector, clean_label))

training_data = new_training_data
test_data = new_test_data

print("Data reshaped successfully!")
print("New training data shape:", training_data[0][0].shape)
print("New label shape:", training_data[0][1].shape)

import numpy as np

# 重新生成最标准的数据
def format_data(X, y):
    result = []
    for i in range(len(X)):
        # 强制把图片变成 (784, 1) 的列向量
        img_vector = X[i].reshape(-1, 1)
        
        # 强制把标签变成 (10, 1) 的 one-hot 列向量
        label_vector = np.zeros((10, 1))
        # 确保取得的是纯数字
        label_num = int(y[i])
        label_vector[label_num] = 1.0
        
        result.append((img_vector, label_vector))
    return result

# 假设你之前的 X_train, y_train, X_test, y_test 还在内存里
# 如果不在，你需要重新运行加载数据的代码
print("Formatting training data...")
training_data = format_data(X_train, y_train)
print("Formatting test data...")
test_data = format_data(X_test, y_test)

# 切分验证集
validation_data = training_data[:10000]
training_data = training_data[10000:]

print("Data formatted successfully!")
print("Training data size:", len(training_data))
print("Validation data size:", len(validation_data))
print("Test data size:", len(test_data))

In [ ]:
net = Network([784, 30, 10])

In [ ]:
net.SGD(training_data, 5, 10, 3.0, test_data = test_data)